In [1]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [2]:
import pandas as pd
import numpy as np
import warnings

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.utils import resample
from sklearn.model_selection import StratifiedKFold, cross_validate

warnings.filterwarnings('ignore')


In [3]:
class RotationForestClassifier(ClassifierMixin, BaseEstimator):
    
    # Etiqueta absoluta para que Scikit-Learn JAMÁS lo trate como regresor
    _estimator_type = "classifier"
    
    def __init__(self, n_estimators=50, max_features_per_subset=15, random_state=None):
        self.n_estimators = n_estimators
        self.max_features_per_subset = max_features_per_subset
        self.random_state = random_state
        
    def fit(self, X, y):
        np.random.seed(self.random_state)
        self.models = []
        self.rotation_matrices = []
        # Definición obligatoria para clasificadores en scikit-learn
        self.classes_ = np.unique(y) 
        
        n_features = X.shape[1]
        
        for i in range(self.n_estimators):
            features = np.random.permutation(n_features)
            subsets = [features[j:j + self.max_features_per_subset] 
                       for j in range(0, n_features, self.max_features_per_subset)]
            
            rotation_matrix = np.zeros((n_features, n_features))
            
            for subset in subsets:
                X_subset = X[:, subset]
                # Remuestreo (Bootstrap)
                X_sample = resample(X_subset, n_samples=int(X.shape[0]*0.75), random_state=self.random_state) 
                
                pca = PCA()
                pca.fit(X_sample)
                
                for j, feature_idx in enumerate(subset):
                    rotation_matrix[feature_idx, subset] = pca.components_[:, j]
                    
            X_rotated = np.dot(X, rotation_matrix)
            
            tree = DecisionTreeClassifier(random_state=self.random_state)
            tree.fit(X_rotated, y)
            
            self.models.append(tree)
            self.rotation_matrices.append(rotation_matrix)
            
        return self
    
    def predict_proba(self, X):
        probas = np.zeros((X.shape[0], len(self.classes_)))
        for tree, rot_mat in zip(self.models, self.rotation_matrices):
            X_rotated = np.dot(X, rot_mat)
            probas += tree.predict_proba(X_rotated)
        return probas / self.n_estimators
        
    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

    # Método de seguridad para validadores de scikit-learn 1.4+
    def _more_tags(self):
        return {"estimator_type": "classifier"}


In [4]:

# 2. Carga y preprocesamiento

df = pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/dataset.csv', sep='\t')

df_pivot = df.pivot(index='Paciente', columns='Bacteria_Limpia', values='AbundanciaTaxonomica')
df_pivot = df_pivot.fillna(0) 

labels = df[['Paciente', 'Diagnosticos']].drop_duplicates().set_index('Paciente')
data = df_pivot.join(labels)

X = data.drop(columns=['Diagnosticos']).values
y = data['Diagnosticos'].values

# 3. Entrenamiento y Evaluación
rf = RotationForestClassifier(n_estimators=50, max_features_per_subset=15, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Ejecutamos el cross validation con n_jobs=1 para evitar errores de paralelización ocultos
results = cross_validate(rf, X, y, cv=cv, scoring=['accuracy', 'f1', 'roc_auc'], n_jobs=1)

# 4. Resultados finales
print("--- MÉTRICAS DEL ROTATION FOREST ---")
print(f"Accuracy : {np.mean(results['test_accuracy']):.4f} ± {np.std(results['test_accuracy']):.4f}")
print(f"F1 Score : {np.mean(results['test_f1']):.4f} ± {np.std(results['test_f1']):.4f}")
print(f"ROC AUC  : {np.mean(results['test_roc_auc']):.4f} ± {np.std(results['test_roc_auc']):.4f}")

--- MÉTRICAS DEL ROTATION FOREST ---
Accuracy : 0.8187 ± 0.0543
F1 Score : 0.8539 ± 0.0421
ROC AUC  : 0.9079 ± 0.0499


In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
import warnings

warnings.filterwarnings('ignore')

# 1. Cargar y pivotear los datos (Melt -> Matriz ancha)
df = pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/dataset.csv', sep='\t')
df_pivot = df.pivot(index='Paciente', columns='Bacteria_Limpia', values='AbundanciaTaxonomica').fillna(0)

# Unir con la columna Diagnosticos
labels = df[['Paciente', 'Diagnosticos']].drop_duplicates().set_index('Paciente')
data = df_pivot.join(labels)

X_df = data.drop(columns=['Diagnosticos'])
X = X_df.values
y = data['Diagnosticos'].values
nombres_bacterias = X_df.columns

# los pacientes con TEA son 0 y los de HC son 1

# 2. Entrenar y validar el Random Forest
# class_weight='balanced' es importante para datos de salud
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', max_depth=5, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

resultados = cross_validate(rf, X, y, cv=cv, scoring=['accuracy', 'f1', 'roc_auc'], n_jobs=1)

print("--- MÉTRICAS DEL RANDOM FOREST ---")
print(f"Accuracy : {np.mean(resultados['test_accuracy']):.4f} ± {np.std(resultados['test_accuracy']):.4f}")
print(f"F1 Score : {np.mean(resultados['test_f1']):.4f} ± {np.std(resultados['test_f1']):.4f}")
print(f"ROC AUC  : {np.mean(resultados['test_roc_auc']):.4f} ± {np.std(resultados['test_roc_auc']):.4f}")

# 3. Extraer Interpretabilidad Biológica (Feature Importance)
# Ajustamos sobre todo el dataset para extraer la firma bacteriana final
rf.fit(X, y)
importancias = rf.feature_importances_

df_importancias = pd.DataFrame({
    'Bacteria': nombres_bacterias,
    'Importancia_Predictiva': importancias
}).sort_values(by='Importancia_Predictiva', ascending=False)

# 4. Calcular el perfil clínico (Aumentado vs Disminuido en TEA)
medias = data.groupby('Diagnosticos').mean().T
medias.columns = ['Media_HC', 'Media_TEA']
medias['Diferencia'] = medias['Media_TEA'] - medias['Media_HC']
medias['Estado_en_TEA'] = np.where(medias['Diferencia'] > 0, 'Aumentada', 'Disminuida')

# Unir la importancia matemática con la dirección biológica
df_final = df_importancias.merge(medias[['Estado_en_TEA']], left_on='Bacteria', right_index=True)

print("\n--- TOP 20 BACTERIAS CLAVE PARA EL DIAGNÓSTICO ---")
# Filtramos bacterias con importancia 0 (las que el RF ignoró por completo)
df_final = df_final[df_final['Importancia_Predictiva'] > 0]
print(df_final.head(20).to_string(index=False))

--- MÉTRICAS DEL RANDOM FOREST ---
Accuracy : 0.8440 ± 0.0674
F1 Score : 0.8716 ± 0.0543
ROC AUC  : 0.9254 ± 0.0619

--- TOP 20 BACTERIAS CLAVE PARA EL DIAGNÓSTICO ---
                       Bacteria  Importancia_Predictiva Estado_en_TEA
                    Collinsella                0.046392     Aumentada
     [Eubacterium]_hallii_group                0.039006     Aumentada
    Erysipelotrichaceae_UCG-003                0.037443     Aumentada
   [Ruminococcus]_torques_group                0.033457     Aumentada
                   Agathobacter                0.028133     Aumentada
                   Faecalitalea                0.027610    Disminuida
                 Parasutterella                0.027086    Disminuida
                      Dialister                0.026260     Aumentada
[Ruminococcus]_gauvreauii_group                0.025979     Aumentada
   Lachnospiraceae_ND3007_group                0.022992    Disminuida
                        Sarcina                0.021771    Dis

In [6]:

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    recall_score,
    f1_score
)
import warnings

warnings.filterwarnings('ignore')

# ============================================================
# 1. CARGAR Y PREPARAR LOS DATOS
# ============================================================

df = pd.read_csv(
    '/kaggle/input/datasets/nico14h/metagenomica/dataset.csv',
    sep='\t'
)

df_pivot = df.pivot(
    index='Paciente',
    columns='Bacteria_Limpia',
    values='AbundanciaTaxonomica'
).fillna(0)

labels = (
    df[['Paciente', 'Diagnosticos']]
    .drop_duplicates()
    .set_index('Paciente')
)

data = df_pivot.join(labels)

X_df = data.drop(columns=['Diagnosticos'])
X = X_df.values
y = data['Diagnosticos'].astype(int).values

nombres_bacterias = X_df.columns

# TEA = 0 (POSITIVA)
# HC  = 1 (NEGATIVA)

assert set(np.unique(y)) == {0, 1}, \
    "Las clases deben ser TEA=0 y HC=1"

print(f"Total de pacientes: {len(y)}")
print(f"Pacientes TEA: {np.sum(y == 0)}")
print(f"Pacientes HC: {np.sum(y == 1)}")
print(f"Total de bacterias: {X.shape[1]}")


# ============================================================
# 2. DEFINIR LAS MÉTRICAS
# ============================================================

def calcular_metricas(y_real, y_pred, prob_tea):

    # ROC-AUC: TEA (0) como clase positiva
    y_binario = (y_real == 0).astype(int)

    auc = roc_auc_score(y_binario, prob_tea)

    # Balanced accuracy
    bal_acc = balanced_accuracy_score(y_real, y_pred)

    # Sensibilidad: proporción de TEA detectados
    sensibilidad = recall_score(
        y_real, y_pred,
        pos_label=0,
        zero_division=0
    )

    # Especificidad: proporción de HC detectados
    especificidad = recall_score(
        y_real, y_pred,
        pos_label=1,
        zero_division=0
    )

    # F1-score de TEA
    f1 = f1_score(
        y_real, y_pred,
        pos_label=0,
        zero_division=0
    )

    return {
        'ROC-AUC': auc,
        'Balanced Accuracy': bal_acc,
        'Sensibilidad (TEA)': sensibilidad,
        'Especificidad (HC)': especificidad,
        'F1-score (TEA)': f1
    }


# ============================================================
# 3. RANDOM FOREST + VALIDACIÓN CRUZADA DE 5 FOLDS
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

resultados_folds = []

# Predicciones out-of-fold: cada paciente se predice
# únicamente con un modelo que NO lo utilizó para entrenar.
predicciones_oof = np.empty(len(y), dtype=int)
probabilidades_oof = np.empty(len(y), dtype=float)

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    rf = RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        max_depth=5,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    # Predicciones de clase
    y_pred = rf.predict(X_test)

    # Probabilidad de TEA (clase 0), NO de HC
    indice_tea = np.where(rf.classes_ == 0)[0][0]
    prob_tea = rf.predict_proba(X_test)[:, indice_tea]

    # Guardar predicciones out-of-fold
    predicciones_oof[test_idx] = y_pred
    probabilidades_oof[test_idx] = prob_tea

    metricas = calcular_metricas(y_test, y_pred, prob_tea)
    metricas['Fold'] = fold
    resultados_folds.append(metricas)

df_resultados = pd.DataFrame(resultados_folds).set_index('Fold')


# ============================================================
# 4. RESULTADOS DE LOS 5 FOLDS
# ============================================================

print("\n========== RESULTADOS POR FOLD ==========")
print(df_resultados.round(4).to_string())

print("\n========== RESUMEN RANDOM FOREST ==========")

resumen = pd.DataFrame({
    'Media': df_resultados.mean(),
    'Desviacion_Estandar': df_resultados.std(ddof=1)
})

for metrica, fila in resumen.iterrows():
    print(
        f"{metrica:25s}: "
        f"{fila['Media']:.4f} ± "
        f"{fila['Desviacion_Estandar']:.4f}"
    )


# ============================================================
# 5. INTERVALOS DE CONFIANZA DEL 95% - BOOTSTRAP OOF
# ============================================================

# Bootstrap estratificado sobre predicciones out-of-fold.
# Proporciona IC aproximados; NO reentrena los modelos.

n_bootstrap = 2000
rng = np.random.default_rng(42)

indices_tea = np.where(y == 0)[0]
indices_hc = np.where(y == 1)[0]

metricas_oof = calcular_metricas(
    y, predicciones_oof, probabilidades_oof
)

valores_bootstrap = {
    metrica: [] for metrica in metricas_oof
}

for _ in range(n_bootstrap):

    muestra_tea = rng.choice(
        indices_tea, size=len(indices_tea), replace=True
    )

    muestra_hc = rng.choice(
        indices_hc, size=len(indices_hc), replace=True
    )

    indices = np.concatenate([muestra_tea, muestra_hc])

    metricas_boot = calcular_metricas(
        y[indices],
        predicciones_oof[indices],
        probabilidades_oof[indices]
    )

    for metrica, valor in metricas_boot.items():
        valores_bootstrap[metrica].append(valor)

print("\n========== IC 95% (BOOTSTRAP OOF) ==========")

for metrica, valores in valores_bootstrap.items():

    ic_inf, ic_sup = np.percentile(valores, [2.5, 97.5])

    print(
        f"{metrica:25s}: "
        f"{metricas_oof[metrica]:.4f} "
        f"[{ic_inf:.4f}, {ic_sup:.4f}]"
    )


# ============================================================
# 6. INTERPRETABILIDAD BIOLÓGICA
# ============================================================

# Entrenamiento final con todos los pacientes.
# SOLO para obtener importancia de las bacterias.
rf_final = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    max_depth=5,
    random_state=42,
    n_jobs=-1
)

rf_final.fit(X, y)

importancias = rf_final.feature_importances_

df_importancias = pd.DataFrame({
    'Bacteria': nombres_bacterias,
    'Importancia_Predictiva': importancias
}).sort_values(
    by='Importancia_Predictiva',
    ascending=False
)


# ============================================================
# 7. PERFIL BIOLÓGICO TEA VS HC
# ============================================================

# CORRECCIÓN:
# Diagnosticos 0 = TEA
# Diagnosticos 1 = HC

medias = data.groupby('Diagnosticos').mean().T

medias = medias.rename(columns={
    0: 'Media_TEA',
    1: 'Media_HC'
})

medias['Diferencia'] = (
    medias['Media_TEA'] - medias['Media_HC']
)

medias['Estado_en_TEA'] = np.select(
    [
        medias['Diferencia'] > 0,
        medias['Diferencia'] < 0
    ],
    [
        'Aumentada',
        'Disminuida'
    ],
    default='Sin cambio'
)

df_final = df_importancias.merge(
    medias[['Estado_en_TEA']],
    left_on='Bacteria',
    right_index=True
)

df_final = df_final[
    df_final['Importancia_Predictiva'] > 0
]

print("\n========== TOP 20 BACTERIAS ==========")
print(df_final.head(20).to_string(index=False))


Total de pacientes: 121
Pacientes TEA: 50
Pacientes HC: 71
Total de bacterias: 152

========== RESULTADOS POR FOLD ==========
      ROC-AUC  Balanced Accuracy  Sensibilidad (TEA)  Especificidad (HC)  F1-score (TEA)
Fold                                                                                    
1      0.8200             0.7000                 0.6              0.8000          0.6316
2      0.9714             0.8643                 0.8              0.9286          0.8421
3      1.0000             0.9000                 0.8              1.0000          0.8889
4      0.9071             0.8286                 0.8              0.8571          0.8000
5      0.9286             0.8643                 0.8              0.9286          0.8421

========== RESUMEN RANDOM FOREST ==========
ROC-AUC                  : 0.9254 ± 0.0692
Balanced Accuracy        : 0.8314 ± 0.0777
Sensibilidad (TEA)       : 0.7600 ± 0.0894
Especificidad (HC)       : 0.9029 ± 0.0765
F1-score (TEA)           : 0.8009 

In [7]:
#Inicio del Extra Tree


import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    recall_score,
    f1_score
)
import warnings

warnings.filterwarnings('ignore')

# ============================================================
# 1. CARGAR Y PREPARAR LOS DATOS
# ============================================================

df = pd.read_csv(
    '/kaggle/input/datasets/nico14h/metagenomica/dataset.csv',
    sep='\t'
)

df_pivot = df.pivot(
    index='Paciente',
    columns='Bacteria_Limpia',
    values='AbundanciaTaxonomica'
).fillna(0)

labels = (
    df[['Paciente', 'Diagnosticos']]
    .drop_duplicates()
    .set_index('Paciente')
)

data = df_pivot.join(labels)

X_df = data.drop(columns=['Diagnosticos'])
X = X_df.values
y = data['Diagnosticos'].astype(int).values

nombres_bacterias = X_df.columns

# TEA = 0 (POSITIVA)
# HC  = 1 (NEGATIVA)

assert set(np.unique(y)) == {0, 1}, \
    "Las clases deben ser TEA=0 y HC=1"

print(f"Total de pacientes: {len(y)}")
print(f"Pacientes TEA: {np.sum(y == 0)}")
print(f"Pacientes HC: {np.sum(y == 1)}")
print(f"Total de bacterias: {X.shape[1]}")


# ============================================================
# 2. DEFINIR LAS MÉTRICAS
# ============================================================

def calcular_metricas(y_real, y_pred, prob_tea):

    # ROC-AUC: TEA (0) como clase positiva
    y_binario = (y_real == 0).astype(int)

    auc = roc_auc_score(y_binario, prob_tea)

    # Balanced accuracy
    bal_acc = balanced_accuracy_score(y_real, y_pred)

    # Sensibilidad: proporción de TEA detectados
    sensibilidad = recall_score(
        y_real, y_pred,
        pos_label=0,
        zero_division=0
    )

    # Especificidad: proporción de HC detectados
    especificidad = recall_score(
        y_real, y_pred,
        pos_label=1,
        zero_division=0
    )

    # F1-score de TEA
    f1 = f1_score(
        y_real, y_pred,
        pos_label=0,
        zero_division=0
    )

    return {
        'ROC-AUC': auc,
        'Balanced Accuracy': bal_acc,
        'Sensibilidad (TEA)': sensibilidad,
        'Especificidad (HC)': especificidad,
        'F1-score (TEA)': f1
    }


# ============================================================
# 3. RANDOM FOREST + VALIDACIÓN CRUZADA DE 5 FOLDS
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

resultados_folds = []

# Predicciones out-of-fold: cada paciente se predice
# únicamente con un modelo que NO lo utilizó para entrenar.
predicciones_oof = np.empty(len(y), dtype=int)
probabilidades_oof = np.empty(len(y), dtype=float)

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    rf = ExtraTreesClassifier(
        n_estimators=100,
        class_weight='balanced',
        max_depth=5,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    # Predicciones de clase
    y_pred = rf.predict(X_test)

    # Probabilidad de TEA (clase 0), NO de HC
    indice_tea = np.where(rf.classes_ == 0)[0][0]
    prob_tea = rf.predict_proba(X_test)[:, indice_tea]

    # Guardar predicciones out-of-fold
    predicciones_oof[test_idx] = y_pred
    probabilidades_oof[test_idx] = prob_tea

    metricas = calcular_metricas(y_test, y_pred, prob_tea)
    metricas['Fold'] = fold
    resultados_folds.append(metricas)

df_resultados = pd.DataFrame(resultados_folds).set_index('Fold')


# ============================================================
# 4. RESULTADOS DE LOS 5 FOLDS
# ============================================================

print("\n========== RESULTADOS POR FOLD ==========")
print(df_resultados.round(4).to_string())

print("\n========== RESUMEN EXTRA TREE ==========")

resumen = pd.DataFrame({
    'Media': df_resultados.mean(),
    'Desviacion_Estandar': df_resultados.std(ddof=1)
})

for metrica, fila in resumen.iterrows():
    print(
        f"{metrica:25s}: "
        f"{fila['Media']:.4f} ± "
        f"{fila['Desviacion_Estandar']:.4f}"
    )


# ============================================================
# 5. INTERVALOS DE CONFIANZA DEL 95% - BOOTSTRAP OOF
# ============================================================

# Bootstrap estratificado sobre predicciones out-of-fold.
# Proporciona IC aproximados; NO reentrena los modelos.

n_bootstrap = 2000
rng = np.random.default_rng(42)

indices_tea = np.where(y == 0)[0]
indices_hc = np.where(y == 1)[0]

metricas_oof = calcular_metricas(
    y, predicciones_oof, probabilidades_oof
)

valores_bootstrap = {
    metrica: [] for metrica in metricas_oof
}

for _ in range(n_bootstrap):

    muestra_tea = rng.choice(
        indices_tea, size=len(indices_tea), replace=True
    )

    muestra_hc = rng.choice(
        indices_hc, size=len(indices_hc), replace=True
    )

    indices = np.concatenate([muestra_tea, muestra_hc])

    metricas_boot = calcular_metricas(
        y[indices],
        predicciones_oof[indices],
        probabilidades_oof[indices]
    )

    for metrica, valor in metricas_boot.items():
        valores_bootstrap[metrica].append(valor)

print("\n========== IC 95% (BOOTSTRAP OOF) ==========")

for metrica, valores in valores_bootstrap.items():

    ic_inf, ic_sup = np.percentile(valores, [2.5, 97.5])

    print(
        f"{metrica:25s}: "
        f"{metricas_oof[metrica]:.4f} "
        f"[{ic_inf:.4f}, {ic_sup:.4f}]"
    )


# ============================================================
# 6. INTERPRETABILIDAD BIOLÓGICA
# ============================================================

# Entrenamiento final con todos los pacientes.
# SOLO para obtener importancia de las bacterias.
rf_final = ExtraTreesClassifier(
    n_estimators=100,
    class_weight='balanced',
    max_depth=5,
    random_state=42,
    n_jobs=-1
)

rf_final.fit(X, y)

importancias = rf_final.feature_importances_

df_importancias = pd.DataFrame({
    'Bacteria': nombres_bacterias,
    'Importancia_Predictiva': importancias
}).sort_values(
    by='Importancia_Predictiva',
    ascending=False
)


# ============================================================
# 7. PERFIL BIOLÓGICO TEA VS HC
# ============================================================

# CORRECCIÓN:
# Diagnosticos 0 = TEA
# Diagnosticos 1 = HC

medias = data.groupby('Diagnosticos').mean().T

medias = medias.rename(columns={
    0: 'Media_TEA',
    1: 'Media_HC'
})

medias['Diferencia'] = (
    medias['Media_TEA'] - medias['Media_HC']
)

medias['Estado_en_TEA'] = np.select(
    [
        medias['Diferencia'] > 0,
        medias['Diferencia'] < 0
    ],
    [
        'Aumentada',
        'Disminuida'
    ],
    default='Sin cambio'
)

df_final = df_importancias.merge(
    medias[['Estado_en_TEA']],
    left_on='Bacteria',
    right_index=True
)

df_final = df_final[
    df_final['Importancia_Predictiva'] > 0
]

print("\n========== TOP 20 BACTERIAS ==========")
print(df_final.head(20).to_string(index=False))


Total de pacientes: 121
Pacientes TEA: 50
Pacientes HC: 71
Total de bacterias: 152

========== RESULTADOS POR FOLD ==========
      ROC-AUC  Balanced Accuracy  Sensibilidad (TEA)  Especificidad (HC)  F1-score (TEA)
Fold                                                                                    
1      0.8733             0.8000                 0.8              0.8000          0.7619
2      0.9857             0.9143                 0.9              0.9286          0.9000
3      1.0000             0.9000                 0.8              1.0000          0.8889
4      0.9000             0.8286                 0.8              0.8571          0.8000
5      0.9214             0.8786                 0.9              0.8571          0.8571

========== RESUMEN EXTRA TREE ==========
ROC-AUC                  : 0.9361 ± 0.0548
Balanced Accuracy        : 0.8643 ± 0.0484
Sensibilidad (TEA)       : 0.8400 ± 0.0548
Especificidad (HC)       : 0.8886 ± 0.0772
F1-score (TEA)           : 0.8416 ± 0

In [8]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from boruta import BorutaPy
import warnings

warnings.filterwarnings('ignore')

# 1. Transformación de la matriz Melt a Matriz Ancha
df = pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/dataset.csv', sep='\t')
df_pivot = df.pivot(index='Paciente', columns='Bacteria_Limpia', values='AbundanciaTaxonomica').fillna(0)

# Unir con la etiqueta de diagnóstico (1 para TEA, 0 para HC)
labels = df[['Paciente', 'Diagnosticos']].drop_duplicates().set_index('Paciente')
data = df_pivot.join(labels)

# 2. Preparación estricta para Boruta (Requiere arrays de NumPy)
X_df = data.drop(columns=['Diagnosticos'])
X = X_df.values
y = data['Diagnosticos'].values
nombres_bacterias = X_df.columns.values

# 3. Configuración del núcleo predictivo
# max_depth=5 es recomendado al usar Boruta para evitar sobreajuste en los árboles
rf = RandomForestClassifier(n_jobs=-1, class_weight='balanced', max_depth=5, random_state=42)

# 4. Inicialización y ajuste de Boruta
# verbose=2 imprimirá el progreso de las iteraciones en la consola
boruta_selector = BorutaPy(rf, n_estimators='auto', verbose=2, random_state=42, max_iter=100)

print("Entrenando Boruta para clasificar TEA vs HC...")
boruta_selector.fit(X, y)

# 5. Estructuración de los biomarcadores extraídos
resultados = pd.DataFrame({
    'Bacteria': nombres_bacterias,
    'Ranking': boruta_selector.ranking_,
    'Estado': ['Confirmado' if r == 1 else 'Tentativo' if r == 2 else 'Rechazado' 
               for r in boruta_selector.ranking_]
})

# Filtrar para mostrar solo las bacterias que el modelo considera útiles
bacterias_importantes = resultados[resultados['Estado'].isin(['Confirmado', 'Tentativo'])].sort_values(by=['Ranking', 'Bacteria'])

print("\n--- BIOMARCADORES CRÍTICOS PARA LA CLASIFICACIÓN ---")
print(bacterias_importantes.to_string(index=False))

Entrenando Boruta para clasificar TEA vs HC...
Iteration: 	1 / 100
Confirmed: 	0
Tentative: 	152
Rejected: 	0
Iteration: 	2 / 100
Confirmed: 	0
Tentative: 	152
Rejected: 	0
Iteration: 	3 / 100
Confirmed: 	0
Tentative: 	152
Rejected: 	0
Iteration: 	4 / 100
Confirmed: 	0
Tentative: 	152
Rejected: 	0
Iteration: 	5 / 100
Confirmed: 	0
Tentative: 	152
Rejected: 	0
Iteration: 	6 / 100
Confirmed: 	0
Tentative: 	152
Rejected: 	0
Iteration: 	7 / 100
Confirmed: 	0
Tentative: 	152
Rejected: 	0
Iteration: 	8 / 100
Confirmed: 	14
Tentative: 	25
Rejected: 	113
Iteration: 	9 / 100
Confirmed: 	14
Tentative: 	25
Rejected: 	113
Iteration: 	10 / 100
Confirmed: 	14
Tentative: 	25
Rejected: 	113
Iteration: 	11 / 100
Confirmed: 	14
Tentative: 	25
Rejected: 	113
Iteration: 	12 / 100
Confirmed: 	16
Tentative: 	18
Rejected: 	118
Iteration: 	13 / 100
Confirmed: 	16
Tentative: 	18
Rejected: 	118
Iteration: 	14 / 100
Confirmed: 	16
Tentative: 	18
Rejected: 	118
Iteration: 	15 / 100
Confirmed: 	16
Tentative: 	18
R

In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesClassifier
from boruta import BorutaPy
import warnings

warnings.filterwarnings('ignore')

# 1. Transformación de la matriz Melt a Matriz Ancha
df = pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/dataset.csv', sep='\t')
df_pivot = df.pivot(index='Paciente', columns='Bacteria_Limpia', values='AbundanciaTaxonomica').fillna(0)

# Unir con la etiqueta de diagnóstico (1 para TEA, 0 para HC)
labels = df[['Paciente', 'Diagnosticos']].drop_duplicates().set_index('Paciente')
data = df_pivot.join(labels)

# 2. Preparación estricta para Boruta
X_df = data.drop(columns=['Diagnosticos'])
X = X_df.values
y = data['Diagnosticos'].values
nombres_bacterias = X_df.columns.values

# 3. Configuración del núcleo predictivo con Extra Trees
et = ExtraTreesClassifier(n_jobs=-1, class_weight='balanced', max_depth=5, random_state=42)


# 4. Inicialización y ajuste de Boruta
boruta_selector = BorutaPy(et, n_estimators='auto', verbose=2, random_state=42, max_iter=100)

print("Entrenando Boruta con Extra Trees para clasificar TEA vs HC...")
boruta_selector.fit(X, y)

# 5. Estructuración de los biomarcadores extraídos
resultados = pd.DataFrame({
    'Bacteria': nombres_bacterias,
    'Ranking': boruta_selector.ranking_,
    'Estado': ['Confirmado' if r == 1 else 'Tentativo' if r == 2 else 'Rechazado' 
               for r in boruta_selector.ranking_]
})

# Filtrar para mostrar solo las bacterias útiles
bacterias_importantes = resultados[resultados['Estado'].isin(['Confirmado', 'Tentativo'])].sort_values(by=['Ranking', 'Bacteria'])

print("\n--- BIOMARCADORES CRÍTICOS PARA LA CLASIFICACIÓN (EXTRA TREES) ---")
print(bacterias_importantes.to_string(index=False))

Entrenando Boruta con Extra Trees para clasificar TEA vs HC...
Iteration: 	1 / 100
Confirmed: 	0
Tentative: 	152
Rejected: 	0
Iteration: 	2 / 100
Confirmed: 	0
Tentative: 	152
Rejected: 	0
Iteration: 	3 / 100
Confirmed: 	0
Tentative: 	152
Rejected: 	0
Iteration: 	4 / 100
Confirmed: 	0
Tentative: 	152
Rejected: 	0
Iteration: 	5 / 100
Confirmed: 	0
Tentative: 	152
Rejected: 	0
Iteration: 	6 / 100
Confirmed: 	0
Tentative: 	152
Rejected: 	0
Iteration: 	7 / 100
Confirmed: 	0
Tentative: 	152
Rejected: 	0
Iteration: 	8 / 100
Confirmed: 	12
Tentative: 	33
Rejected: 	107
Iteration: 	9 / 100
Confirmed: 	12
Tentative: 	33
Rejected: 	107
Iteration: 	10 / 100
Confirmed: 	12
Tentative: 	33
Rejected: 	107
Iteration: 	11 / 100
Confirmed: 	12
Tentative: 	33
Rejected: 	107
Iteration: 	12 / 100
Confirmed: 	16
Tentative: 	29
Rejected: 	107
Iteration: 	13 / 100
Confirmed: 	16
Tentative: 	27
Rejected: 	109
Iteration: 	14 / 100
Confirmed: 	16
Tentative: 	27
Rejected: 	109
Iteration: 	15 / 100
Confirmed: 	16
